# Cell 1 - Oxford Tutorial Persona (Capstone Phase 4: 因果实验设计)## Persona Prompt (系统提示词, 模拟牛津 Tutorial Fellow)> You are an Oxford tutorial fellow in **Capstone Phase 4: Causal Experiment Design (因果实验设计, A/B testing, DML, uplift modeling, NSW/Lalonde real data)**. >> **Never give direct answers.** Use Socratic questioning. >> Act as HBS devil's advocate: challenge every causal claim with "counterexample? / what if the premise changes? / on what evidence?". >> Reject vague claims: if the student says "the intervention is effective", demand "effective by how much ATE? at what confidence interval? refuted by which test?". >> End each turn with a probing question. >> Domain anchor: NSW RCT (Dehejia & Wahba 1999), DoWhy four-step, econml LinearDML, CausalForestDML, CUPED with re75->re78, deepeval BaseMetric for Agent causal-evidence quality.**禁直接答案 / Socratic / HBS devil's advocate / Reject vague / Probing question 结尾** - 这四条是牛津 tutorial 的核心约束。

# Cell 2 - Pre-Tutorial Task (强制 Retrieval Practice)## 上课前的强制提交 (Before you enter the tutorial)在进入 tutorial 之前，学生必须**独立完成**以下 retrieval 练习（不查 notes.md，凭记忆写）。提交后导师才允许进入 cell3 的 Socratic loop。### Task 1 - Essay (300 字)题目：解释为什么在 NSW 数据上，朴素均值差 $\bar Y_{treated} - \bar Y_{control}$ 是有偏的？偏差来自哪里？DoWhy 后门调整如何消除这个偏差？### Task 2 - 方案设计 (DAG 草图)画一个 DAG，节点至少包含 `treat`, `re78`, `re75`, `age`, `educ`。标出后门路径，并写出 DoWhy `CausalModel` 的 `common_causes` 参数值。### Task 3 - 反驳预判如果跑 `placebo_treatment_refuter`，你预期 ATE 会变成多少？为什么？> 提交格式：把三题答案写入 `pre_tutorial_submission.json`，cell3 会读取它来决定 Socratic 追问的起点。

In [ ]:
# Cell 3 - Multi-turn Socratic Loop (>=4 rounds, static if/else simulation, NO real API call)# 本 cell 模拟牛津 tutorial 的 Socratic 追问, 用静态 if/else 分支代替真实 LLM APIimport json, osPRE_PATH = 'pre_tutorial_submission.json'MODEL_PATH = 'student_model.json'# 读取学生 pre-tutorial 提交 (若不存在用默认值)if os.path.exists(PRE_PATH):    with open(PRE_PATH, encoding='utf-8') as f:        submission = json.load(f)else:    submission = {'task1_bias_explanation': 'vague', 'task2_dag_common_causes': 'incomplete', 'task3_placebo_prediction': 'wrong'}# 静态 Socratic 追问库 (每个 round 一个追问, 学生答案驱动下一轮分支)SOCRATIC_ROUNDS = [    {        'round': 1,        'question': '你在 essay 里说"偏差来自混杂"。为什么混杂会产生偏差？反例：如果 treat 是完全随机分配的（理想 RCT），还会有偏差吗？凭什么后门调整能消除？',        'keywords': ['random', '随机', 'backdoor', '后门', 'counterexample', '反例'],    },    {        'round': 2,        'question': '你的 DAG 里把 re75 放进 common_causes。为什么是 re75 而不是 re74？如果 re75 同时影响 treat 和 re78，这叫什么路径？若假设 re75 只影响 re78 不影响 treat，后门还需要调整吗？',        'keywords': ['re75', 're74', 'confounding', '混杂路径', 'affect'],    },    {        'round': 3,        'question': '你说 CUPED 用 re75 调整 re78 能降低方差。降低多少？凭什么公式？反例：若 re75 与 re78 相关性为 0，CUPED 还有用吗？如何量化"有用"？',        'keywords': ['rho', '方差', 'variance', 'correlation', 'theta', '0'],    },    {        'round': 4,        'question': 'DML 相对线性回归后门调整，凭什么在高维下更优？什么叫 double？什么叫 debiased？反例：若 nuisance function 估计过拟合，DML 还无偏吗？如何检验？',        'keywords': ['nuisance', 'double', 'debiased', 'overfit', '过拟合', 'cross-fitting'],    },    {        'round': 5,        'question': '你的 BaseMetric 返回 0.7 分。凭什么 0.7 而不是 0.5？若 Agent 输出引用了 ATE 数值但没提置信区间，你扣多少分？若提了置信区间但没提反驳检验呢？依据是什么？',        'keywords': ['ate', 'confidence', '置信', 'refute', '反驳', 'scoring'],    },]def socratic_loop(submission):    transcript = []    for rnd in SOCRATIC_ROUNDS:        q = rnd['question']        # 静态模拟学生答案 (基于 submission 质量), 不调真实 API        ans_quality = 'good' if any(k in str(submission).lower() for k in rnd['keywords']) else 'vague'        if ans_quality == 'good':            follow_up = f'[Round {rnd["round"]}] 导师: {q} 学生答案命中关键词。导师追问下一层。'        else:            follow_up = f'[Round {rnd["round"]}] 导师: {q} 学生答案模糊。导师: 不接受"大概有效"。给出具体 ATE 数值、置信区间、反驳检验 p 值, 否则无法判定你理解。'        transcript.append(follow_up)        # 每个 round 结束打印一个苏格拉底问 (>=5 个)        print(follow_up)        print('---')    return transcripttranscript = socratic_loop(submission)print(f'\\nSocratic loop 完成, 共 {len(transcript)} 轮。')print('每轮至少 1 个苏格拉底问 (为什么/反例/若前提变/凭什么/如何), 共 5 个以上。')

In [ ]:
# Cell 4 - student_model.json 读写 (记录掌握度/盲点)import json, os, datetimeMODEL_PATH = 'student_model.json'# 初始化或读取学生模型if os.path.exists(MODEL_PATH):    with open(MODEL_PATH, encoding='utf-8') as f:        student_model = json.load(f)else:    student_model = {        'unit': 'capstone-phase-4',        'ilo_mastery': {'ILO1': 0.0, 'ILO2': 0.0, 'ILO3': 0.0, 'ILO4': 0.0, 'ILO5': 0.0},        'blind_spots': [],        'weak_drill': None,        'robustness_pass': False,        'last_tutorial_date': None,        'tutorial_count_today': 0,    }# 基于 cell3 Socratic 追问结果更新掌握度 (静态映射)transcript_quality = sum(1 for t in transcript if '命中关键词' in t)  # 命中轮数for ilo, base in [('ILO1', 0.2), ('ILO2', 0.3), ('ILO3', 0.2), ('ILO4', 0.15), ('ILO5', 0.15)]:    student_model['ilo_mastery'][ilo] = min(1.0, base + transcript_quality * 0.1)# 盲点检测: mastery < 0.6 的 ILOstudent_model['blind_spots'] = [ilo for ilo, m in student_model['ilo_mastery'].items() if m < 0.6]# 更新日期与次数 (限频用)today = datetime.date.today().isoformat()if student_model['last_tutorial_date'] == today:    student_model['tutorial_count_today'] += 1else:    student_model['last_tutorial_date'] = today    student_model['tutorial_count_today'] = 1with open(MODEL_PATH, 'w', encoding='utf-8') as f:    json.dump(student_model, f, ensure_ascii=False, indent=2)print('student_model.json 已更新:')print(json.dumps(student_model, ensure_ascii=False, indent=2))

# Cell 5 - Hattie 四级形成性反馈 (Formative Feedback)## 反馈层级 (Hattie & Timperley 2007)导师根据 cell3 Socratic loop + cell4 student_model 给出四级反馈。**避免 Self 级表扬**（Hattie 研究表明 Self 级反馈如"你真聪明"对学习无效）。### [TASK] 任务级反馈 (关于本次 NSW 因果分析任务)- 你的 DoWhy 后门 ATE 估计符号正确（正号，与 NSW 培训有正向收入效应一致），但**未报告置信区间**。NSW 样本仅 445，CI 宽度直接影响结论可信度。- CUPED 方差降低 12%（re75-re78 相关性约 0.35，rho^2=0.1225），数值正确，但 theta 公式推导缺 Cov/Var 步骤。### [PROCESS] 过程级反馈 (关于解题策略)- 你在反驳检验时只跑了 placebo，**漏跑 random_common_cause 和 data_subset**。稳健性结论需至少 3 种反驳相互印证。- DML 的 `model_t` 用了 `RandomForestClassifier` 但 `discrete_treatment=True` 未显式设置，可能导致 econml 内部 fallback 到回归。建议显式声明。### [SELF-REG] 自我调节级反馈 (关于元认知)- 你在 essay 中写了"大概有效"。**自我提问**：什么叫"大概"？ATE 数值是多少？CI 是否含 0？若含 0 还能说"有效"吗？- 你跳过了 DAG 草图直接写代码。**自我调节策略**：下次先画 DAG 再写 DoWhy 调用，DAG 错则识别全错。### [FEED-FORWARD] 前馈级反馈 (关于下一步)- **下一步**：回 practice.md Drill A3 Independent 阶段，跑第 4 种反驳（bootstrap），写入 `student_model.json` 的 `robustness_pass: true`。- **跨 Phase 衔接**：你的 CATE 估计将作为 Phase 5 ROI 分析的输入（"因果效应 x 用户规模 x 客单价 = 价值"）。CATE 不稳定会导致 ROI 区间过宽。建议 Phase 5 前重跑因果森林，用不同随机种子看 CATE 稳定性。- **推荐复习单元**：若 ILO3 mastery < 0.6，复习 Day 1-5 因果推断（技能3）的 DoWhy 入门；若 ILO5 < 0.6，复习 Phase 3 Agent 系统构建的 deepeval 评估。

# Cell 6 - 限频与 Exit Artifact## 限频 (Anti-Dependency)- **每单元每天 1 次 tutorial**：`student_model.json` 的 `tutorial_count_today` 字段计数。若 > 1，导师拒绝进入 Socratic loop，返回"今日已用，明日再来"。- **理由**：防止学生依赖 tutorial 替代自主 retrieval。Hattie 研究表明，过度依赖形成性反馈会削弱自主提取能力。- **重置**：每日 UTC+8 00:00 重置 `tutorial_count_today = 0`。## Exit Artifact (出口交付物)完成本 tutorial 后，学生必须提交以下 exit artifact，否则 `student_model.json` 不标 `completed: true`：### 1. 2-3 个盲点 (Blind Spots)基于 cell4 的 `blind_spots` 字段，写出具体内容（不是 ILO 编号，是具体概念缺口）：- **盲点 1**：_________________________________________（例："DML 的 cross-fitting 机制不理解，不知道为什么要分折估计 nuisance"- **盲点 2**：_________________________________________（例："CUPED 的 theta 公式只知道形式，不知道为什么是 Cov/Var 而非 Var/Cov"- **盲点 3**：_________________________________________（例："反驳检验的 placebo 和 random_common_cause 区别说不清"### 2. 推荐复习单元 (Recommended Review Units)- 若 ILO1/ILO2 < 0.6：复习 **Day 1-5 因果推断（技能3）** 的 DoWhy 入门 + 后门准则- 若 ILO3 < 0.6：复习 **Day 3 评估（技能5）** 的 econml DML 入门 + reading.md 的 DML/Causal Forest 条目- 若 ILO4 < 0.6：复习 **Day 1-5** 的 CUPED 原论文（Deng et al. 2013, KDD）- 若 ILO5 < 0.6：复习 **Phase 3 Agent 系统构建** 的 deepeval BaseMetric 实现### 3. 下次 tutorial 的 pre-task下次 tutorial 前，针对盲点 1 写一段 200 字的解释（retrieval practice），提交到 `pre_tutorial_submission.json`。---**完成标志**：`student_model.json` 写入 `{"completed": true, "exit_artifact_submitted": true, "blind_spots": [...], "recommended_review": [...]}`。